## Módulo 1 — Diseño de Codificador y Decodificador

Este notebook implementa la primera etapa de un sistema de comunicación LoRa: la conversión entre una secuencia de bits y los símbolos numéricos que luego se modularán como chirps (eso se aborda en el Módulo 2).

## Objetivo

LoRa no transmite bits de forma directa. Agrupa los bits en bloques de tamaño `SF` (Spreading Factor) y cada bloque se representa como un único número entero (el "símbolo"). En este módulo se implementan:

1. **Codificador**: bits → símbolos
2. **Decodificador**: símbolos → bits
3. **Verificación**: que decodificar lo codificado devuelva los bits originales
4. **Cálculo de BER** (Bit Error Rate): debe ser 0 en esta instancia

## 1. Fundamento teórico

Cada símbolo LoRa puede tomar valores entre `0` y `2^SF - 1`. Por ejemplo, con `SF = 4`:

- Cada símbolo representa 4 bits.
- Hay `2^4 = 16` símbolos posibles (0 a 15).

**Convención usada MSB primero:** dentro de cada bloque de SF bits, el primer bit es el más significativo.

Ejemplo con SF = 4 y bloque `1011`:

```
bit:    1   0   1   1
peso:   8   4   2   1
```

Símbolo = 1·8 + 0·4 + 1·2 + 1·1 = **11**


## 2. Librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 3. Codificador (bits → símbolos)

**Idea paso a paso:**

1. Verificar que la cantidad de bits sea múltiplo de `SF` (si no, no se puede agrupar en bloques completos).
2. Recorrer la secuencia de bits de a bloques de tamaño `SF`.
3. Para cada bloque, recorrer sus bits de izquierda a derecha (MSB primero) y acumular el valor decimal usando potencias de 2 decrecientes.
4. Guardar ese valor como el símbolo correspondiente a ese bloque.

Usamos bucles explícitos (`for`) para que cada paso sea visible y fácil de seguir.


In [ ]:
def codificador(bits, SF):
    """
    Convierte una secuencia de bits en símbolos LoRa.

    Parámetros
    ----------
    bits : array-like de 0s y 1s
    SF   : Spreading Factor (cantidad de bits por símbolo)

    Retorna
    -------
    simbolos : np.array de enteros, uno por cada bloque de SF bits
    """
    bits = np.array(bits)

    if len(bits) % SF != 0:
        raise ValueError(
            f"La cantidad de bits ({len(bits)}) debe ser múltiplo de SF ({SF})"
        )

    num_simbolos = len(bits) // SF
    simbolos = np.zeros(num_simbolos, dtype=int)

    # Recorremos cada bloque de SF bits
    for i in range(num_simbolos):
        bloque = bits[i * SF : (i + 1) * SF]

        valor = 0
        # Recorremos los bits del bloque de izquierda a derecha (MSB primero)
        for posicion, bit in enumerate(bloque):
            peso = 2 ** (SF - 1 - posicion)  # potencia de 2 decreciente
            valor += bit * peso

        simbolos[i] = valor

    return simbolos

## 4. Decodificador (símbolos → bits)

**Idea paso a paso:**

1. Para cada símbolo, hay que reconstruir sus `SF` bits.
2. Recorremos las potencias de 2 de mayor a menor (de `2^(SF-1)` a `2^0`).
3. En cada paso preguntamos: "¿el símbolo tiene esta potencia de 2 'encendida'?" usando división entera y resto, que es equivalente a un corrimiento de bits (`>>`) seguido de una máscara (`& 1`), pero lo escribimos de forma explícita para que se entienda sin operadores bit a bit.


In [ ]:
def decodificador(simbolos, SF):
    """
    Convierte una secuencia de símbolos LoRa de vuelta a bits.

    Parámetros
    ----------
    simbolos : array-like de enteros (cada uno entre 0 y 2^SF - 1)
    SF       : Spreading Factor

    Retorna
    -------
    bits : np.array de 0s y 1s
    """
    simbolos = np.array(simbolos, dtype=int)
    num_bits = len(simbolos) * SF
    bits = np.zeros(num_bits, dtype=int)

    for i, simbolo in enumerate(simbolos):
        valor_restante = int(simbolo)

        # Recorremos las posiciones de bit de la más significativa a la menos significativa
        for posicion in range(SF):
            peso = 2 ** (SF - 1 - posicion)

            if valor_restante >= peso:
                bits[i * SF + posicion] = 1
                valor_restante -= peso
            else:
                bits[i * SF + posicion] = 0

    return bits

## 5. Prueba manual (paso a paso, sin aleatoriedad)

Antes de probar con datos aleatorios, conviene verificar el ejemplo manual que vimos en la teoría:
bloque `1011` con `SF = 4` debería dar símbolo `11`, y al decodificar `11` debería devolver `1011`.


In [ ]:
SF_prueba = 4
bits_prueba = np.array([1, 0, 1, 1])

simbolo_obtenido = codificador(bits_prueba, SF_prueba)
print("Bits de entrada:      ", bits_prueba)
print("Símbolo esperado:      11")
print("Símbolo obtenido:     ", simbolo_obtenido)

bits_recuperados = decodificador(simbolo_obtenido, SF_prueba)
print("\nBits recuperados:     ", bits_recuperados)
print("¿Coinciden con los originales?", np.array_equal(bits_prueba, bits_recuperados))

## 6. Prueba con varios bloques

Probamos ahora con más de un símbolo en la misma llamada, para confirmar que el recorrido por bloques funciona bien.


In [ ]:
SF = 7
num_simbolos_prueba = 5
np.random.seed(42)  # reproducibilidad

bits_random = np.random.randint(0, 2, num_simbolos_prueba * SF)
print("Bits generados:", bits_random)

simbolos = codificador(bits_random, SF)
print("\nSímbolos obtenidos:", simbolos)
print("Rango válido: 0 a", 2**SF - 1)

bits_dec = decodificador(simbolos, SF)
print("\n¿Los bits decodificados son iguales a los originales?",
      np.array_equal(bits_random, bits_dec))

## 7. Verificación exhaustiva

En vez de probar con un puñado de casos aleatorios, conviene comprobar que el codificador-decodificador funciona para **todos los símbolos posibles** de un SF dado. Esto da más confianza de que no hay un caso límite (por ejemplo, el símbolo 0 o el símbolo máximo) que falle silenciosamente.


In [ ]:
def verificar_todos_los_simbolos(SF):
    """
    Codifica y decodifica cada símbolo posible (de 0 a 2^SF - 1)
    representado como sus SF bits, y verifica que el resultado
    sea consistente en ambos sentidos.
    """
    N = 2 ** SF
    errores = 0

    for simbolo_original in range(N):
        # Símbolo -> bits (usamos el propio decodificador para generar el patrón de bits)
        bits_de_ese_simbolo = decodificador([simbolo_original], SF)

        # bits -> símbolo, usando el codificador
        simbolo_reconstruido = codificador(bits_de_ese_simbolo, SF)[0]

        if simbolo_reconstruido != simbolo_original:
            errores += 1
            print(f"❌ Falla en símbolo {simbolo_original}: se obtuvo {simbolo_reconstruido}")

    if errores == 0:
        print(f"✅ Los {N} símbolos posibles (SF={SF}) se codifican y decodifican correctamente.")
    else:
        print(f"Se encontraron {errores} errores de {N} símbolos.")

verificar_todos_los_simbolos(SF=7)

## 8. Cálculo de BER (Bit Error Rate)

Esta función no se usa todavía en este módulo (porque acá no hay canal ni ruido), pero se prepara desde ya porque se va a reutilizar en los módulos 3, 4 y 5, donde sí va a haber errores de bit producidos por el canal.

**BER = (cantidad de bits distintos) / (cantidad total de bits)**


In [ ]:
def calcular_ber(bits_tx, bits_rx):
    """
    Calcula la tasa de error de bit (BER) comparando los bits transmitidos
    contra los bits recibidos/decodificados.
    """
    bits_tx = np.array(bits_tx)
    bits_rx = np.array(bits_rx)

    if len(bits_tx) != len(bits_rx):
        raise ValueError("Los vectores de bits deben tener la misma longitud")

    errores = np.sum(bits_tx != bits_rx)
    ber = errores / len(bits_tx)
    return ber

## 9. Prueba de la función de BER

Probamos `calcular_ber` con un caso sin errores y con un caso donde introducimos errores manualmente (todavía no hay canal real, eso viene en el Módulo 3).


In [ ]:
# Caso 1: sin errores
ber_sin_errores = calcular_ber(bits_random, bits_dec)
print("BER sin errores:", ber_sin_errores)

# Caso 2: forzamos 3 errores de bit manualmente para probar la métrica
bits_con_errores = bits_dec.copy()
indices_con_error = [0, 5, 10]
bits_con_errores[indices_con_error] = 1 - bits_con_errores[indices_con_error]  # invierte esos bits

ber_con_errores = calcular_ber(bits_random, bits_con_errores)
print("BER con 3 errores introducidos:", ber_con_errores)
print("BER esperado:", 3 / len(bits_random))

## 10. Visualización: bits vs símbolos

Una forma útil de entender el "spreading" es graficar los bits originales junto con los símbolos resultantes, para ver visualmente cómo cada bloque de SF bits colapsa en un solo número.


In [ ]:
SF_vis = 4
bits_vis = np.array([1,0,1,1, 0,0,0,1, 1,1,1,1, 0,0,0,0])
simbolos_vis = codificador(bits_vis, SF_vis)

fig, axs = plt.subplots(2, 1, figsize=(9, 5))

axs[0].stem(bits_vis)
axs[0].set_title("Bits originales")
axs[0].set_xlabel("Índice de bit")
axs[0].set_ylabel("Valor")
axs[0].set_yticks([0, 1])

# Marcamos los límites de cada bloque de SF bits
for i in range(1, len(bits_vis) // SF_vis):
    axs[0].axvline(i * SF_vis - 0.5, color='red', linestyle='--', alpha=0.5)

axs[1].stem(simbolos_vis)
axs[1].set_title(f"Símbolos resultantes (SF={SF_vis}, cada uno representa {SF_vis} bits)")
axs[1].set_xlabel("Índice de símbolo")
axs[1].set_ylabel("Valor del símbolo")

plt.tight_layout()
plt.show()

## 11. Conclusiones del módulo

- Se implementó el codificador y decodificador de bits a símbolos LoRa usando la convención MSB-primero.
- Se verificó la consistencia codificador-decodificador para **todos** los símbolos posibles con SF=7, no solo casos aleatorios.
- Se preparó la función `calcular_ber`, que se reutilizará en los módulos siguientes una vez que se introduzca ruido de canal.
- Estas funciones (`codificador`, `decodificador`, `calcular_ber`) son la base sobre la que se construye el Módulo 2 (Waveform Former), donde estos símbolos se convertirán en señales chirp.
